# Prothom Alo — Full Daily Edition Crawler

This notebook crawls `prothomalo.com` itself, discovers collection/topic/section pages and pagination, checks every discovered article's actual publication date in Asia/Dhaka, and creates a clickable EPUB with the complete article body.

In [ ]:
!pip -q install requests beautifulsoup4 ebooklib lxml pillow

import os
import re
import json
import html
import time
import hashlib
import io
from concurrent.futures import ThreadPoolExecutor, as_completed

import datetime as dt
from collections import deque, defaultdict
from urllib.parse import urljoin, urlparse, urldefrag, parse_qs, urlencode

import requests
from bs4 import BeautifulSoup
from ebooklib import epub
from PIL import Image, ImageOps

from google.colab import drive
drive.mount("/content/drive")

# ================================================================
# SETTINGS
# ================================================================

BASE_URL = "https://www.prothomalo.com/"
OUTPUT_DIR = "/content/drive/MyDrive/Prothom_Alo_EPUB"

# For normal daily use:
TARGET_DATE = dt.datetime.now(
    dt.timezone(dt.timedelta(hours=6))
).date()

# For testing the 7 August 2026 edition, use:
# TARGET_DATE = dt.date(2026, 8, 8)

MAX_LISTING_PAGES = 500
MAX_ARTICLE_URLS = 5000
MAX_ARTICLES = 0                 # 0 = unlimited

MAX_DEPTH = 5
REQUEST_DELAY = 0.10
TIMEOUT = 25

# ================================================================
# IMAGE OPTIMIZATION
# ================================================================
IMAGE_MAX_WIDTH = 1100
IMAGE_MAX_HEIGHT = 1100
IMAGE_JPEG_QUALITY = 68
IMAGE_TARGET_KB = 220
IMAGE_WORKERS = 8

# Keep one relevant article image per article.
MAX_IMAGES_PER_ARTICLE = 1


MAX_EMPTY_PAGES_IN_A_ROW = 5

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/150.0 Safari/537.36"
    ),
    "Accept-Language": "bn-BD,bn;q=0.95,en;q=0.8",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

try:
    from zoneinfo import ZoneInfo
    DHAKA = ZoneInfo("Asia/Dhaka")
except Exception:
    DHAKA = dt.timezone(dt.timedelta(hours=6))

# ================================================================
# SECTION MAP
# ================================================================

SECTION_MAP = {
    "bangladesh": "বাংলাদেশ",
    "politics": "রাজনীতি",
    "world": "আন্তর্জাতিক",
    "international": "আন্তর্জাতিক",
    "business": "বাণিজ্য",
    "economy": "বাণিজ্য",
    "sports": "খেলা",
    "entertainment": "বিনোদন",
    "lifestyle": "জীবনযাপন",
    "opinion": "মতামত",
    "technology": "প্রযুক্তি",
    "career": "চাকরি",
    "chakri": "চাকরি",
    "feature": "ফিচার",
    "religion": "ধর্ম",
    "photo": "ছবিতে",
    "photos": "ছবিতে",
    "education": "শিক্ষা",
    "science": "বিজ্ঞান",
    "health": "স্বাস্থ্য",
    "jobs": "চাকরি",
}

SECTION_ORDER = [
    "বাংলাদেশ",
    "রাজনীতি",
    "আন্তর্জাতিক",
    "বাণিজ্য",
    "মতামত",
    "খেলা",
    "বিনোদন",
    "জীবনযাপন",
    "প্রযুক্তি",
    "চাকরি",
    "শিক্ষা",
    "স্বাস্থ্য",
    "বিজ্ঞান",
    "ফিচার",
    "ধর্ম",
    "ছবিতে",
    "অন্যান্য",
]

# These are NOT article sections. They are useful listing/collection roots.
LISTING_ROOTS = (
    "/collection/",
    "/topic/",
    "/category/",
    "/section/",
    "/archive",
    "/latest",
    "/trending",
    "/most-read",
)

BLOCKED_FIRST_SEGMENTS = {
    "search", "login", "signup", "subscribe", "subscription",
    "about", "contact", "terms", "privacy", "privacy-policy",
    "author", "authors", "tag", "tags", "api", "assets", "static",
    "cdn", "amp", "print", "share", "epaper", "rss"
}

BAD_EXTENSIONS = (
    ".jpg", ".jpeg", ".png", ".gif", ".webp", ".svg",
    ".pdf", ".mp4", ".mp3", ".css", ".js", ".xml", ".json"
)

session = requests.Session()
session.headers.update(HEADERS)



# ================================================================
# URL UTILITIES
# ================================================================

def normalize_url(url):
    if not url:
        return None

    url = url.strip()

    if url.startswith("//"):
        url = "https:" + url

    url = urljoin(BASE_URL, url)
    url, _ = urldefrag(url)

    p = urlparse(url)

    if p.scheme not in ("http", "https"):
        return None

    if p.netloc.lower() not in {
        "www.prothomalo.com",
        "prothomalo.com"
    }:
        return None

    # Keep query because ?page=2 etc. can be meaningful.
    path = p.path or "/"

    # Normalize trailing slash except root.
    if path != "/":
        path = path.rstrip("/")

    return (
        "https://www.prothomalo.com"
        + path
        + (f"?{p.query}" if p.query else "")
    )

def path_parts(url):
    return [
        x for x in urlparse(url).path.strip("/").split("/")
        if x
    ]

def first_segment(url):
    p = path_parts(url)
    return p[0].lower() if p else ""

def is_same_domain(url):
    if not url:
        return False
    return urlparse(url).netloc.lower() in {
        "www.prothomalo.com", "prothomalo.com"
    }

def is_bad_asset(url):
    path = urlparse(url).path.lower()
    return any(path.endswith(x) for x in BAD_EXTENSIONS)

def is_blocked(url):
    if not is_same_domain(url) or is_bad_asset(url):
        return True

    first = first_segment(url)

    if first in BLOCKED_FIRST_SEGMENTS:
        return True

    return False

def is_listing_url(url):
    if not url or is_blocked(url):
        return False

    p = urlparse(url)
    path = p.path.lower()
    query = p.query.lower()

    if any(path.startswith(x) for x in LISTING_ROOTS):
        return True

    # Pagination is almost always a listing page.
    if re.search(r"(^|&)page=\d+", query):
        return True

    if re.search(r"(^|&)p=\d+", query):
        return True

    # Common section roots.
    parts = path_parts(url)

    if len(parts) == 1 and first_segment(url) in SECTION_MAP:
        return True

    return False

def is_probable_article(url):
    if not url or is_blocked(url):
        return False

    p = urlparse(url)
    path = p.path.strip("/")

    if not path:
        return False

    query = p.query.lower()

    # Query pagination URLs are listing pages.
    if re.search(r"(^|&)page=\d+", query):
        return False

    if re.search(r"(^|&)p=\d+", query):
        return False

    if is_listing_url(url):
        return False

    parts = path_parts(url)

    # Article URLs generally have at least section + slug/id.
    return len(parts) >= 2

# ================================================================
# FETCH
# ================================================================

def fetch_text(url):
    try:
        time.sleep(REQUEST_DELAY)

        r = session.get(
            url,
            timeout=TIMEOUT,
            allow_redirects=True
        )
        r.raise_for_status()

        return r.text, r.url

    except Exception as e:
        print(f"    FETCH ERROR: {url} :: {e}")
        return None, url

def get_soup(url):
    text, final_url = fetch_text(url)

    if not text:
        return None, final_url

    try:
        return BeautifulSoup(text, "lxml"), final_url
    except Exception:
        return BeautifulSoup(text, "html.parser"), final_url

# ================================================================
# DATE PARSING
# ================================================================

def parse_datetime(value):
    if not value:
        return None

    value = str(value).strip()

    # ISO.
    try:
        x = value.replace("Z", "+00:00")
        d = dt.datetime.fromisoformat(x)

        if d.tzinfo is None:
            d = d.replace(tzinfo=dt.timezone.utc)

        return d

    except Exception:
        pass

    # Date + optional time.
    m = re.search(
        r"(\d{4}-\d{2}-\d{2})"
        r"(?:[T ](\d{2}:\d{2}(?::\d{2})?))?",
        value
    )

    if m:
        try:
            t = m.group(2) or "00:00:00"

            if len(t) == 5:
                t += ":00"

            return dt.datetime.fromisoformat(
                f"{m.group(1)}T{t}"
            ).replace(tzinfo=dt.timezone.utc)

        except Exception:
            pass

    return None

def find_date_in_jsonld(soup):
    for script in soup.find_all(
        "script",
        type="application/ld+json"
    ):
        raw = script.string or script.get_text(
            " ",
            strip=False
        )

        if not raw:
            continue

        for key in (
            "datePublished",
            "dateCreated",
            "dateModified"
        ):
            m = re.search(
                rf'"{key}"\s*:\s*"([^"]+)"',
                raw
            )

            if m:
                d = parse_datetime(m.group(1))
                if d:
                    return d

    return None

def get_published_datetime(soup):
    # JSON-LD is usually the cleanest.
    d = find_date_in_jsonld(soup)

    if d:
        return d

    meta_candidates = [
        ("meta", {"property": "article:published_time"}),
        ("meta", {"name": "article:published_time"}),
        ("meta", {"property": "og:article:published_time"}),
        ("meta", {"property": "datePublished"}),
        ("meta", {"name": "datePublished"}),
    ]

    for tag, attrs in meta_candidates:
        x = soup.find(tag, attrs=attrs)

        if x:
            d = parse_datetime(
                x.get("content")
                or x.get("datetime")
            )

            if d:
                return d

    for t in soup.find_all("time"):
        d = parse_datetime(
            t.get("datetime")
            or t.get("content")
        )

        if d:
            return d

    return None

def is_target_date(d):
    if not d:
        return False

    if d.tzinfo is None:
        d = d.replace(tzinfo=dt.timezone.utc)

    return d.astimezone(DHAKA).date() == TARGET_DATE

# ================================================================
# HTML / JSON LINK DISCOVERY
# ================================================================

def extract_all_links(soup):
    links = set()

    if not soup:
        return links

    for a in soup.find_all("a", href=True):
        u = normalize_url(a.get("href"))

        if u and is_same_domain(u):
            links.add(u)

    # Also inspect common data-* URL attributes.
    for tag in soup.find_all(True):
        for attr in (
            "data-href",
            "data-url",
            "data-link",
            "data-load-more",
            "data-next",
            "data-next-url"
        ):
            value = tag.get(attr)

            if value:
                u = normalize_url(value)

                if u and is_same_domain(u):
                    links.add(u)

    # Extract absolute Prothom Alo URLs embedded in scripts.
    html_text = str(soup)

    patterns = [
        r'https://www\.prothomalo\.com/[A-Za-z0-9_%./?=&\-]+',
        r'https://prothomalo\.com/[A-Za-z0-9_%./?=&\-]+',
    ]

    for pattern in patterns:
        for match in re.findall(pattern, html_text):
            u = normalize_url(match)

            if u and is_same_domain(u):
                links.add(u)

    return links

def find_pagination_links(soup):
    out = set()

    if not soup:
        return out

    for a in soup.find_all("a", href=True):
        text = a.get_text(" ", strip=True).lower()
        href = normalize_url(a.get("href"))

        if not href:
            continue

        if not is_same_domain(href):
            continue

        if (
            "load more" in text
            or "লোড মোর" in text
            or "আরও" in text
            or "next" in text
            or "পরের" in text
            or "পরবর্তী" in text
        ):
            out.add(href)
            continue

        q = urlparse(href).query.lower()

        if "page=" in q or re.search(r"(^|&)p=\d+", q):
            out.add(href)

    # data-next / data-load-more
    for tag in soup.find_all(True):
        for attr in (
            "data-next",
            "data-next-url",
            "data-load-more",
            "data-url"
        ):
            value = tag.get(attr)

            if not value:
                continue

            href = normalize_url(value)

            if href:
                out.add(href)

    return out

# ================================================================
# SEED DISCOVERY
# ================================================================

def get_seed_urls():
    seeds = {
        normalize_url(BASE_URL),
        normalize_url("/collection/latest"),
        normalize_url("/collection/trending"),
        normalize_url("/collection/latest-video"),
    }

    # Homepage will expose many collection/topic/section URLs.
    soup, _ = get_soup(BASE_URL)

    if soup:
        for u in extract_all_links(soup):
            low = u.lower()

            if (
                "/collection/" in low
                or "/topic/" in low
                or "/category/" in low
                or "/section/" in low
                or first_segment(u) in SECTION_MAP
            ):
                seeds.add(u)

    return {
        x for x in seeds
        if x and is_same_domain(x)
    }

# ================================================================
# LISTING CRAWLER
# ================================================================

def crawl_listings():
    seeds = get_seed_urls()

    print("=" * 75)
    print("LISTING SEEDS:", len(seeds))
    for x in sorted(seeds):
        print(" ", x)
    print("=" * 75)

    queue = deque(
        (u, 0)
        for u in seeds
    )

    visited = set()
    article_urls = set()

    empty_pages = 0

    while queue and len(visited) < MAX_LISTING_PAGES:

        url, depth = queue.popleft()

        if url in visited:
            continue

        visited.add(url)

        print(
            f"[LISTING {len(visited)}/{MAX_LISTING_PAGES}] "
            f"d={depth}: {url}"
        )

        soup, final_url = get_soup(url)

        if not soup:
            continue

        links = extract_all_links(soup)

        before = len(article_urls)

        for link in links:

            if is_probable_article(link):
                article_urls.add(link)

                if len(article_urls) >= MAX_ARTICLE_URLS:
                    break

        added = len(article_urls) - before

        # Follow pagination/load-more links.
        next_links = find_pagination_links(soup)

        for nxt in next_links:
            if nxt not in visited:
                queue.append((nxt, depth + 1))

        # Follow collection/topic/section/listing pages discovered
        # inside this page.
        if depth < MAX_DEPTH:

            for link in links:

                if is_listing_url(link):
                    if link not in visited:
                        queue.append((link, depth + 1))

        if added == 0:
            empty_pages += 1
        else:
            empty_pages = 0

        # Do not stop merely because one page is empty. Stop only after
        # several consecutive empty listing pages and the queue is small.
        if (
            empty_pages >= MAX_EMPTY_PAGES_IN_A_ROW
            and len(queue) < 5
        ):
            break

        if len(article_urls) >= MAX_ARTICLE_URLS:
            break

    print()
    print("=" * 75)
    print("LISTING CRAWL FINISHED")
    print("Visited listing pages:", len(visited))
    print("Candidate article URLs:", len(article_urls))
    print("=" * 75)

    return article_urls

# ================================================================
# ARTICLE EXTRACTION
# ================================================================

def extract_title(soup):
    for attrs in (
        {"property": "og:title"},
        {"name": "twitter:title"},
    ):
        x = soup.find("meta", attrs=attrs)

        if x and x.get("content"):
            return x["content"].strip()

    h1 = soup.find("h1")

    if h1:
        x = h1.get_text(" ", strip=True)

        if x:
            return x

    if soup.title:
        x = soup.title.get_text(
            " ",
            strip=True
        )

        x = re.sub(
            r"\s*\|\s*প্রথম আলো.*$",
            "",
            x
        ).strip()

        if x:
            return x

    return None

def jsonld_objects(soup):
    objects = []

    for script in soup.find_all(
        "script",
        type="application/ld+json"
    ):
        raw = script.string or script.get_text(
            " ",
            strip=False
        )

        if not raw:
            continue

        try:
            data = json.loads(raw)

            if isinstance(data, list):
                objects.extend(data)

            elif isinstance(data, dict):
                objects.append(data)

        except Exception:
            continue

    return objects

def extract_article_body_from_jsonld(soup):
    # Search all JSON-LD objects recursively for articleBody.
    def walk(x):
        if isinstance(x, dict):
            if x.get("articleBody"):
                return x["articleBody"]

            for v in x.values():
                r = walk(v)
                if r:
                    return r

        elif isinstance(x, list):
            for v in x:
                r = walk(v)
                if r:
                    return r

        return None

    body = walk(jsonld_objects(soup))

    if body:
        return body.strip()

    return None

def clean_body_dom(soup):
    # Work on a clone-like parse to avoid damaging title/image extraction.
    raw = str(soup)
    s = BeautifulSoup(raw, "lxml")

    for tag in s.find_all([
        "script", "style", "noscript", "iframe",
        "nav", "header", "footer", "aside",
        "form", "button"
    ]):
        tag.decompose()

    article = s.find("article")

    if article:
        root = article
    else:
        root = s.body or s

    # Find paragraph-dense container.
    best = root
    best_count = 0

    for p in root.find_all("p"):
        parent = p.parent

        if not parent:
            continue

        count = len(
            parent.find_all(
                "p",
                recursive=False
            )
        )

        if count > best_count:
            best = parent
            best_count = count

    parts = []

    for el in best.find_all(
        ["p", "blockquote", "ul", "ol"],
        recursive=False
    ):

        text = el.get_text(
            " ",
            strip=True
        )

        if len(text) < 3:
            continue

        # Remove obvious UI boilerplate only.
        low = text.lower()

        bad = (
            "গুগল নিউজ",
            "প্রথম আলোর খবর পেতে",
            "সাবস্ক্রাইব",
            "শেয়ার",
            "কমেন্ট",
            "follow us",
        )

        if any(x in low for x in bad):
            continue

        if el.name == "p":
            parts.append(
                f"<p>{html.escape(text)}</p>"
            )

        elif el.name == "blockquote":
            parts.append(
                "<blockquote>"
                f"{html.escape(text)}"
                "</blockquote>"
            )

        else:
            items = []

            for li in el.find_all(
                "li",
                recursive=False
            ):
                x = li.get_text(
                    " ",
                    strip=True
                )

                if x:
                    items.append(
                        f"<li>{html.escape(x)}</li>"
                    )

            if items:
                parts.append(
                    f"<{el.name}>"
                    + "".join(items)
                    + f"</{el.name}>"
                )

    return "\n".join(parts)

def html_to_epub_body(raw):
    """
    Convert articleBody into clean EPUB HTML without exposing literal
    HTML tags such as <p> to the reader.

    JSON-LD articleBody is normally plain text, but some pages return
    HTML fragments. We parse those fragments instead of html.escape()
    on the whole string.
    """
    if not raw:
        return ""

    raw = html.unescape(str(raw)).strip()

    # If the source contains HTML tags, parse them and keep only useful
    # article structures.
    if re.search(r"<\s*(p|br|blockquote|ul|ol|li|h[1-6])\b", raw, re.I):
        frag = BeautifulSoup(raw, "lxml")
        parts = []

        for el in frag.find_all(["p", "blockquote", "ul", "ol", "h2", "h3", "h4"]):
            if el.name in {"ul", "ol"}:
                items = []
                for li in el.find_all("li", recursive=False):
                    text = li.get_text(" ", strip=True)
                    if text:
                        items.append(f"<li>{html.escape(text)}</li>")
                if items:
                    parts.append(f"<{el.name}>{''.join(items)}</{el.name}>")
                continue

            text = el.get_text(" ", strip=True)
            if not text:
                continue

            if el.name == "blockquote":
                parts.append(f"<blockquote>{html.escape(text)}</blockquote>")
            elif el.name in {"h2", "h3", "h4"}:
                parts.append(f"<{el.name}>{html.escape(text)}</{el.name}>")
            else:
                parts.append(f"<p>{html.escape(text)}</p>")

        if parts:
            return "\n".join(parts)

    # Plain-text fallback.
    raw = re.sub(r"\r\n?", "\n", raw)
    paragraphs = [x.strip() for x in re.split(r"\n\s*\n+", raw) if x.strip()]

    if len(paragraphs) == 1:
        # Some JSON-LD stores one long string with single newlines.
        paragraphs = [x.strip() for x in raw.split("\n") if x.strip()]

    return "\n".join(
        f"<p>{html.escape(p)}</p>"
        for p in paragraphs
    )


def extract_body(soup):
    body = extract_article_body_from_jsonld(soup)

    if body:
        cleaned = html_to_epub_body(body)
        if cleaned:
            return cleaned

    return clean_body_dom(soup)

def normalize_image_url(url, base_url=None):
    """Normalize image URLs without requiring the image host to be www.prothomalo.com."""
    if not url:
        return None

    url = html.unescape(str(url)).strip()

    if url.startswith("//"):
        url = "https:" + url

    if base_url:
        url = urljoin(base_url, url)

    p = urlparse(url)

    if p.scheme not in ("http", "https"):
        return None

    # Ignore SVG/data/blob assets and obvious UI icons.
    path = p.path.lower()
    if path.endswith((".svg", ".gif")):
        return None

    return url


def _image_is_noise(url, img=None):
    """Reject logos, UI graphics, ads, icons and other non-news images."""
    text = (url or "").lower()

    if img is not None:
        attrs = " ".join([
            str(img.get("class") or ""),
            str(img.get("id") or ""),
            str(img.get("alt") or ""),
            str(img.get("title") or ""),
        ]).lower()
        text += " " + attrs

    noise_terms = (
        "logo", "prothomalo-logo", "favicon", "icon", "sprite",
        "google", "preferred", "source", "follow", "subscribe",
        "advert", "ads", "banner", "placeholder", "avatar",
        "author", "share", "facebook", "twitter", "youtube",
        "loading", "default-image", "app-store", "play-store"
    )
    return any(term in text for term in noise_terms)

def _image_candidates_from_node(img):
    candidates = [
        img.get("src"),
        img.get("data-src"),
        img.get("data-lazy-src"),
        img.get("data-original"),
        img.get("data-image"),
    ]

    for attr in ("srcset", "data-srcset", "data-lazy-srcset"):
        value = img.get(attr)
        if value:
            parsed = []
            for item in value.split(","):
                bits = item.strip().split()
                if not bits:
                    continue
                width = 0
                if len(bits) > 1:
                    m = re.search(r"(\d+)w", bits[1])
                    if m:
                        width = int(m.group(1))
                parsed.append((width, bits[0]))
            if parsed:
                parsed.sort(reverse=True)
                candidates.append(parsed[0][1])

    return [x for x in candidates if x]

def _caption_for_image(img):
    parent = img.parent
    if parent:
        fc = parent.find("figcaption")
        if fc:
            return fc.get_text(" ", strip=True)
    return ""

def extract_images(soup):
    """
    Return exactly ONE useful news image.
    Priority:
      1. real image inside article body
      2. article JSON-LD image
      3. og:image fallback
    """
    body_root = (
        soup.select_one("article")
        or soup.select_one("[itemprop='articleBody']")
        or soup.select_one(".article-body")
        or soup
    )

    # First and strongest signal: actual article-body image.
    if body_root:
        for img in body_root.find_all("img"):
            candidates = _image_candidates_from_node(img)

            if _image_is_noise(" ".join(candidates), img):
                continue

            try:
                w = int(img.get("width") or 0)
                h = int(img.get("height") or 0)
                if w and h and (w < 180 or h < 120):
                    continue
            except Exception:
                pass

            for candidate in candidates:
                u = normalize_image_url(candidate, BASE_URL)
                if not u or _image_is_noise(u, img):
                    continue
                return [(u, _caption_for_image(img))]

    # JSON-LD Article/NewsArticle image.
    for obj in jsonld_objects(soup):
        if not isinstance(obj, dict):
            continue

        value = obj.get("image")
        values = value if isinstance(value, list) else [value]

        for item in values:
            if isinstance(item, dict):
                candidates = [item.get("url"), item.get("contentUrl")]
            else:
                candidates = [item]

            for candidate in candidates:
                u = normalize_image_url(candidate, BASE_URL)
                if u and not _image_is_noise(u):
                    return [(u, "")]

    # Final fallback: social preview image.
    for attrs in (
        {"property": "og:image"},
        {"property": "og:image:url"},
        {"name": "twitter:image"},
        {"name": "twitter:image:src"},
    ):
        tag = soup.find("meta", attrs=attrs)
        if tag and tag.get("content"):
            u = normalize_image_url(tag["content"], BASE_URL)
            if u and not _image_is_noise(u):
                return [(u, "")]

    return []

def extract_section(soup, url):
    parts = path_parts(url)

    if parts:
        key = parts[0].lower()

        if key in SECTION_MAP:
            return SECTION_MAP[key]

    # Breadcrumb fallback.
    for tag in soup.find_all(
        ["nav", "ol", "div"],
        class_=re.compile(
            r"breadcrumb",
            re.I
        )
    ):
        text = tag.get_text(
            " ",
            strip=True
        )

        if text:
            last = text.split(">")[-1].strip()

            if len(last) <= 60:
                return last

    return "অন্যান্য"

# ================================================================
# SCRAPE ARTICLE
# ================================================================

def scrape_article(url):
    soup, final_url = get_soup(url)

    if not soup:
        return None

    published = get_published_datetime(soup)

    if not is_target_date(published):
        return None

    title = extract_title(soup)

    if not title:
        return None

    body = extract_body(soup)

    if not body:
        return None

    section = extract_section(
        soup,
        final_url
    )

    return {
        "url": final_url,
        "title": title,
        "section": section,
        "published": published,
        "body": body,
        "images": extract_images(soup),
    }

# ================================================================
# DOWNLOAD IMAGE
# ================================================================

def _optimize_image_bytes(content):
    """Resize and compress image for a much smaller EPUB."""
    try:
        with Image.open(io.BytesIO(content)) as im:
            im = ImageOps.exif_transpose(im)

            # Flatten transparency onto white for JPEG.
            if im.mode in ("RGBA", "LA", "P"):
                bg = Image.new("RGB", im.size, "white")
                if im.mode == "P":
                    im = im.convert("RGBA")
                bg.paste(im, mask=im.getchannel("A") if "A" in im.getbands() else None)
                im = bg
            else:
                im = im.convert("RGB")

            # Resize only when needed; never enlarge.
            im.thumbnail(
                (IMAGE_MAX_WIDTH, IMAGE_MAX_HEIGHT),
                Image.Resampling.LANCZOS
            )

            # Start with configured quality and reduce if the image is large.
            quality = IMAGE_JPEG_QUALITY
            while True:
                out = io.BytesIO()
                im.save(
                    out,
                    format="JPEG",
                    quality=quality,
                    optimize=True,
                    progressive=True
                )
                data = out.getvalue()

                if len(data) <= IMAGE_TARGET_KB * 1024 or quality <= 48:
                    return data

                quality -= 5

    except Exception:
        # If Pillow cannot decode it, keep original bytes as a fallback.
        return content

def download_image(url):
    try:
        # No artificial REQUEST_DELAY here: article crawling can retain its
        # delay, but image downloads are optimized concurrently.
        r = session.get(
            url,
            timeout=TIMEOUT,
            allow_redirects=True,
            headers={
                **HEADERS,
                "Referer": BASE_URL,
                "Accept": "image/avif,image/webp,image/apng,image/jpeg,image/png,*/*;q=0.8",
            },
        )
        r.raise_for_status()

        content = r.content
        if not content:
            return None, None

        optimized = _optimize_image_bytes(content)
        return optimized, (".jpg", "image/jpeg")

    except Exception as e:
        print(f"    IMAGE ERROR: {url} :: {e}")
        return None, None

def download_images_concurrently(urls):
    """Download/optimize unique image URLs concurrently."""
    unique = list(dict.fromkeys(urls))
    results = {}

    if not unique:
        return results

    print(
        f"    Optimizing {len(unique)} image(s) "
        f"with {IMAGE_WORKERS} workers..."
    )

    with ThreadPoolExecutor(max_workers=IMAGE_WORKERS) as executor:
        future_map = {
            executor.submit(download_image, url): url
            for url in unique
        }

        for future in as_completed(future_map):
            url = future_map[future]
            try:
                results[url] = future.result()
            except Exception as e:
                print(f"    IMAGE ERROR: {url} :: {e}")
                results[url] = (None, None)

    return results

# ================================================================
# EPUB
# ================================================================

CSS = """
body {
    font-family: serif;
    line-height: 1.72;
    margin: 5%;
}
.book-title {
    text-align: center;
    font-size: 1.9em;
}
.edition-date {
    text-align: center;
    margin-bottom: 2em;
}
.section-title {
    font-size: 1.35em;
    border-bottom: 1px solid #777;
    padding-bottom: .15em;
    margin-top: 1.5em;
}
.headline {
    margin-bottom: .65em;
    margin-left: .15em;
}
.headline-number {
    display: inline-block;
    min-width: 1.8em;
    font-weight: bold;
}
.article-title {
    font-size: 1.65em;
    line-height: 1.35;
}
.section-label {
    font-size: .85em;
}
.back {
    text-align: right;
    font-size: .85em;
    margin: 1em 0;
}
.article-image {
    max-width: 100%;
    height: auto;
    margin: .8em 0;
}
.caption {
    font-size: .8em;
    font-style: italic;
    margin-top: -.5em;
    margin-bottom: 1em;
}
blockquote {
    margin-left: 1em;
    padding-left: 1em;
}
"""

def build_epub(articles):
    os.makedirs(
        OUTPUT_DIR,
        exist_ok=True
    )

    output = os.path.join(
        OUTPUT_DIR,
        f"prothomalo_{TARGET_DATE.isoformat()}.epub"
    )

    book = epub.EpubBook()

    book.set_identifier(
        f"prothomalo-{TARGET_DATE.isoformat()}"
    )

    book.set_title(
        f"প্রথম আলো — {TARGET_DATE.isoformat()}"
    )

    book.set_language("bn")
    book.add_author("প্রথম আলো")

    css = epub.EpubItem(
        uid="style",
        file_name="style/main.css",
        media_type="text/css",
        content=CSS
    )

    book.add_item(css)

    # Create chapter objects first.
    chapter_map = {}

    for i, article in enumerate(
        articles,
        start=1
    ):
        ch = epub.EpubHtml(
            title=article["title"],
            file_name=f"text/article_{i:04d}.xhtml",
            lang="bn"
        )

        ch.add_item(css)

        chapter_map[
            article["url"]
        ] = ch

    grouped = defaultdict(list)

    for article in articles:
        grouped[
            article["section"]
        ].append(article)

    ordered_sections = (
        SECTION_ORDER
        + [
            s for s in grouped
            if s not in SECTION_ORDER
        ]
    )

    # -------------------- FIRST PAGE / INDEX --------------------

    blocks = []

    for section in ordered_sections:

        items = grouped.get(
            section,
            []
        )

        if not items:
            continue

        numbered = []

        for n, article in enumerate(items, start=1):

            ch = chapter_map[
                article["url"]
            ]

            numbered.append(
                '<p class="headline">'
                f'<span class="headline-number">{n}.</span> '
                f'<a href="{ch.file_name}">'
                f'{html.escape(article["title"])}'
                '</a>'
                '</p>'
            )

        blocks.append(
            f'<h2 class="section-title">'
            f'{html.escape(section)}'
            f'</h2>'
            f'{"".join(numbered)}'
        )

    index_html = f"""
    <html>
    <head>
        <title>সূচিপত্র</title>
        <link rel="stylesheet"
              href="style/main.css"/>
    </head>
    <body>
        <h1 class="book-title">প্রথম আলো</h1>
        <p class="edition-date">
            {TARGET_DATE.strftime("%d-%m-%Y")}
        </p>

        {"".join(blocks)}
    </body>
    </html>
    """

    index = epub.EpubHtml(
        title="সূচিপত্র",
        file_name="index.xhtml",
        lang="bn",
        content=index_html
    )

    index.add_item(css)
    book.add_item(index)

    # -------------------- ARTICLES --------------------

    chapters = []
    image_no = 0

    # Download and optimize all unique article images before building XHTML.
    # This is much faster than doing one HTTP request per article serially.
    all_image_urls = []
    for article in articles:
        for image_url, _caption in article.get("images", [])[:MAX_IMAGES_PER_ARTICLE]:
            all_image_urls.append(image_url)

    image_cache = download_images_concurrently(all_image_urls)

    total_image_bytes = sum(
        len(data)
        for data, _meta in image_cache.values()
        if data
    )

    print(
        f"    Optimized image payload: "
        f"{total_image_bytes / (1024 * 1024):.2f} MB"
    )

    for i, article in enumerate(
        articles,
        start=1
    ):
        ch = chapter_map[
            article["url"]
        ]

        print(
            f"[EPUB {i}/{len(articles)}] "
            f"{article['title']}"
        )

        image_html = ""

        for image_url, caption in article["images"][:MAX_IMAGES_PER_ARTICLE]:

            data, meta = image_cache.get(
                image_url,
                (None, None)
            )

            if not data:
                continue

            image_no += 1

            ext, media = meta

            fname = (
                f"images/img_{image_no:04d}{ext}"
            )

            item = epub.EpubItem(
                uid=f"image_{image_no}",
                file_name=fname,
                media_type=media,
                content=data
            )

            book.add_item(item)

            cap = ""

            if caption:
                cap = (
                    '<p class="caption">'
                    f'{html.escape(caption)}'
                    '</p>'
                )

            image_html += (
                f'<img class="article-image" '
                f'src="../{fname}" alt=""/>'
                f'{cap}'
            )

        article_html = f"""
        <html>
        <head>
            <title>
                {html.escape(article["title"])}
            </title>
            <link rel="stylesheet"
                  href="../style/main.css"/>
        </head>

        <body>

            <p class="back">
                <a href="../index.xhtml">
                    ← সূচিপত্রে ফিরে যান
                </a>
            </p>

            <p class="section-label">
                {html.escape(article["section"])}
            </p>

            <h1 class="article-title">
                {html.escape(article["title"])}
            </h1>

            {image_html}

            {article["body"]}

            <p class="back">
                <a href="../index.xhtml">
                    ↑ সূচিপত্রে ফিরে যান
                </a>
            </p>

        </body>
        </html>
        """

        ch.set_content(
            article_html
        )

        book.add_item(ch)
        chapters.append(ch)

    # -------------------- EPUB TOC --------------------

    toc = []

    for section in ordered_sections:

        items = grouped.get(
            section,
            []
        )

        if not items:
            continue

        toc.append(
            (
                epub.Section(section),
                [
                    chapter_map[
                        a["url"]
                    ]
                    for a in items
                ]
            )
        )

    book.toc = toc

    book.add_item(
        epub.EpubNcx()
    )

    book.add_item(
        epub.EpubNav()
    )

    # The visible first page is the index.
    book.spine = [
        index
    ] + chapters

    epub.write_epub(
        output,
        book
    )

    try:
        size_mb = os.path.getsize(output) / (1024 * 1024)
        print(f"    EPUB SIZE: {size_mb:.2f} MB")
    except Exception:
        pass

    return output

# ================================================================
# MAIN
# ================================================================

def main():
    print()
    print("=" * 75)
    print("PROTHOM ALO FULL DAILY EDITION")
    print("Target date:", TARGET_DATE)
    print("Timezone: Asia/Dhaka")
    print("=" * 75)

    candidate_urls = crawl_listings()

    print()
    print("Now checking every candidate article...")
    print()

    articles = []

    # Do not depend on arbitrary URL ordering.
    # Every candidate is checked.
    for i, url in enumerate(
        sorted(candidate_urls),
        start=1
    ):
        print(
            f"[ARTICLE {i}/{len(candidate_urls)}]"
        )

        item = scrape_article(url)

        if item:
            articles.append(item)

            print(
                "   ✓ TODAY:",
                item["title"]
            )

            if (
                MAX_ARTICLES
                and len(articles) >= MAX_ARTICLES
            ):
                break

    # Deduplicate by final URL.
    unique = []
    seen = set()

    for article in articles:

        key = article["url"]

        if key in seen:
            continue

        seen.add(key)
        unique.append(article)

    # Newest first.
    unique.sort(
        key=lambda x: (
            x["published"]
            or dt.datetime.min.replace(
                tzinfo=dt.timezone.utc
            )
        ),
        reverse=True
    )

    print()
    print("=" * 75)
    print(
        "TODAY'S COMPLETE ARTICLES:",
        len(unique)
    )
    print("=" * 75)

    if not unique:
        print(
            "No articles found for",
            TARGET_DATE
        )
        return None

    output = build_epub(
        unique
    )

    print()
    print("=" * 75)
    print("EPUB READY")
    print("Articles:", len(unique))
    print("File:", output)
    print("=" * 75)

    # Save a simple manifest for debugging.
    manifest = output.replace(
        ".epub",
        "_manifest.json"
    )

    with open(
        manifest,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            [
                {
                    "title": a["title"],
                    "section": a["section"],
                    "url": a["url"],
                    "published": (
                        a["published"].isoformat()
                        if a["published"]
                        else None
                    )
                }
                for a in unique
            ],
            f,
            ensure_ascii=False,
            indent=2
        )

    # Download EPUB automatically in Colab.
    try:
        from google.colab import files
        files.download(output)
    except Exception:
        pass

    return output

# RUN
output_file = main()
